# AWS S3 Vectors: Connect and Test

This notebook verifies your S3 Vectors access end to end: create the client, list vector buckets, and validate index visibility.

<style>
.jp-Notebook .jp-MarkdownCell .jp-RenderedHTMLCommon {
  font-family: "Charter", "Palatino Linotype", "Book Antiqua", Georgia, serif;
  font-size: 17px;
  line-height: 1.7;
}
.jp-Notebook .jp-MarkdownCell .jp-RenderedHTMLCommon h1,
.jp-Notebook .jp-MarkdownCell .jp-RenderedHTMLCommon h2,
.jp-Notebook .jp-MarkdownCell .jp-RenderedHTMLCommon h3 {
  letter-spacing: 0.2px;
  line-height: 1.35;
}
.jp-Notebook .jp-MarkdownCell {
  background: #f8fafc;
  border-left: 4px solid #2563eb;
  border-radius: 6px;
  padding: 4px 10px;
}
</style>

This notebook uses a learning-focused markdown style so explanation cells are easier to scan than code cells.

Set up imports. We only use `boto3` and one helper for optional pretty output.

In [ ]:
import boto3
import json

Define your AWS settings and target vector resources. Keep these values in one place so the rest of the notebook stays clean.

In [ ]:
AWS_ACCESS_KEY_ID = "REPLACE_ME"
AWS_SECRET_ACCESS_KEY = "REPLACE_ME"
AWS_REGION = "us-east-1"

BUCKET_NAME = "airline-policy-vectors-YOURNAME"
VECTOR_INDEX_NAME = "airline-policy-index"

Create a session and open the `s3vectors` client. This is your API door into vector buckets and indexes.

In [ ]:
session = boto3.Session(
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    region_name=AWS_REGION,
)
client = session.client("s3vectors")

print("Connected to s3vectors in", AWS_REGION)

List vector buckets available to your IAM identity.

Analogy: think of this like asking a library system which branches your card can enter before searching for books.

In [ ]:
bucket_response = client.list_vector_buckets()
entries = bucket_response.get("vectorBuckets") or []
bucket_names = [x.get("vectorBucketName") for x in entries if x.get("vectorBucketName")]

print("Vector buckets found:", len(bucket_names))
bucket_names[:10]

Check whether your configured bucket is visible in the returned list. This is a quick signal that account, region, and permissions all line up.

In [ ]:
bucket_visible = BUCKET_NAME in bucket_names
print("Configured bucket visible:", bucket_visible)
if not bucket_visible:
    print("Double-check bucket name, region, and IAM scope.")

List indexes inside your vector bucket and verify your expected index name appears.

In [ ]:
index_response = client.list_indexes(vectorBucketName=BUCKET_NAME)
index_entries = index_response.get("indexes") or []
index_names = [x.get("indexName") for x in index_entries if x.get("indexName")]

print("Indexes found:", len(index_names))
print("Expected index visible:", VECTOR_INDEX_NAME in index_names)
index_names

Optional debug view: inspect a compact JSON payload if you need to troubleshoot what the API returned.

In [ ]:
preview = {
    "vectorBuckets": entries[:3],
    "indexes": index_entries[:3],
}
print(json.dumps(preview, indent=2, default=str))